# 00 · Setup compartido

Este notebook **no se ejecuta solo**: cada notebook lo llama con `%run ./00_Setup`.

| Qué deja listo | Para qué |
|---|---|
| Esquema y volumen en Unity Catalog | Un lugar propio para el laboratorio (`lab_tvmaze_m34`) |
| Rutas y nombres de tablas | Un solo sitio donde cambiarlos |
| Esquema del JSON de TVMaze | Se define una vez y lo usa Auto Loader |
| Funciones del pipeline | `run_bronze()`, `build_silver()`, `build_gold()` y ayudantes |

```
API TVMaze ─► RAW (JSON en el Volumen) ─► BRONZE ─► SILVER ─► GOLD
                        Auto Loader ──┘   CDF+MERGE ┘  CDF+MERGE ┘
```

## 1. Ubicación: catálogo, esquema y volumen

In [0]:
import time, inspect, requests, json, re
from datetime import date, timedelta
from pathlib import Path
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

CATALOG = spark.sql("SELECT current_catalog()").first()[0]   # 'workspace' en la Free Edition
SCHEMA  = "lab_tvmaze_m34"
VOLUME  = "datos"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS `{CATALOG}`.{SCHEMA}.{VOLUME}")
spark.sql(f"USE CATALOG `{CATALOG}`")
spark.sql(f"USE SCHEMA {SCHEMA}")

## 2. Rutas y nombres de tablas

In [0]:
# Archivos (Volumen)
BASE            = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
RAW_PATH        = f"{BASE}/raw/schedule"            # JSON tal cual llega: schedule_AAAA-MM-DD.json
SCHEMA_PATH     = f"{BASE}/autoloader/schema"       # Auto Loader guarda aquí el esquema
CHECKPOINT_PATH = f"{BASE}/autoloader/checkpoint"   # Auto Loader recuerda aquí qué archivos ya leyó

# Tablas (Delta)
BRONZE = "bronze_schedule"      # crudo, tal como llega (Auto Loader)
SILVER = "silver_episodes"      # limpio, una fila por episodio (CDF de bronze + MERGE)
GOLD   = "gold_network_daily"   # episodios por red y día (CDF de silver + MERGE)
STATE  = "pipeline_state"       # hasta qué versión del origen procesó cada capa

## 3. Esquema explícito del JSON

Solo los campos que usaremos. Lo que no está aquí, Auto Loader lo guarda en `_rescued_data`.

In [0]:
ESQUEMA_EPISODIOS = """
    id BIGINT, name STRING, season INT, number INT, type STRING,
    airdate STRING, airtime STRING, airstamp STRING, runtime INT,
    rating STRUCT<average: DOUBLE>,
    _embedded STRUCT<
        show: STRUCT<
            id: BIGINT, name: STRING, type: STRING, language: STRING,
            genres: ARRAY<STRING>, status: STRING, premiered: STRING,
            network: STRUCT<id: BIGINT, name: STRING, country: STRUCT<name: STRING, code: STRING>>,
            webChannel: STRUCT<id: BIGINT, name: STRING, country: STRUCT<name: STRING, code: STRING>>
        >
    >
"""

## 4. Ayudantes: nombres, descarga de la API y versiones

In [0]:
def qualified_name(tabla):
    """Nombre completo de una tabla: catalogo.esquema.tabla"""
    return f"{CATALOG}.{SCHEMA}.{tabla}"


def download_day(fecha):
    """Descarga un día de la API a RAW. Devuelve 'nuevo', 'existía' o 'falló'."""
    destino = Path(RAW_PATH) / f"schedule_{fecha.isoformat()}.json"
    if destino.exists():
        return "existía"
    try:
        r = requests.get("https://api.tvmaze.com/schedule/web", params={"date": fecha.isoformat()}, timeout=30)
        r.raise_for_status()
    except requests.RequestException:
        return "falló"
    destino.parent.mkdir(parents=True, exist_ok=True)
    destino.write_bytes(r.content)          # el JSON exactamente como lo devolvió la API
    time.sleep(0.5)                         # cortesía con la API
    return "nuevo"


def download_days(inicio, dias):
    """Descarga varios días seguidos y resume el resultado."""
    res = {"nuevo": 0, "existía": 0, "falló": 0}
    for i in range(dias):
        res[download_day(inicio + timedelta(days=i))] += 1
    print(f"Descargados: {res['nuevo']} | Ya existían: {res['existía']} | Fallidos: {res['falló']}")


def raw_files():
    """Nombres de los archivos JSON que hay en RAW, ordenados."""
    if not Path(RAW_PATH).exists():
        return []
    return sorted(f.name for f in dbutils.fs.ls(RAW_PATH))


def next_day():
    """El día siguiente al último archivo de RAW: simula que 'llega un día nuevo'."""
    dias = [f for f in raw_files() if re.fullmatch(r"schedule_\d{4}-\d{2}-\d{2}\.json", f)]
    return date.fromisoformat(dias[-1][9:19]) + timedelta(days=1)


def write_correction(episode_id, runtime, nombre):
    """Simula que la fuente reenvía un episodio corregido: un JSON nuevo en RAW con ese episodio y otro runtime."""
    for f in raw_files():
        episodios = json.loads((Path(RAW_PATH) / f).read_text(encoding="utf-8"))
        episodio = next((e for e in episodios if e["id"] == episode_id), None)
        if episodio:
            break
    episodio["runtime"] = runtime
    (Path(RAW_PATH) / f"correccion_{nombre}.json").write_text(json.dumps([episodio]), encoding="utf-8")
    print(f"Archivo correccion_{nombre}.json: episodio {episode_id} con runtime = {runtime}")


def table_version(tabla):
    """Última versión de una tabla Delta."""
    return spark.sql(f"DESCRIBE HISTORY {tabla}").agg(F.max("version")).first()[0]


def show_history(tabla):
    """Historial resumido de una tabla Delta."""
    display(spark.sql(f"DESCRIBE HISTORY {tabla}")
            .select("version", "timestamp", "operation", "operationMetrics")
            .orderBy("version"))

## 5. Auto Loader por lotes: RAW → bronze

Dos funciones separadas, como pide el diseño: una **lee** (detecta archivos nuevos) y otra **escribe** en bronze.
Corre por lotes: `trigger(availableNow=True)` procesa lo pendiente y se detiene.
La escritura es un **upsert**: cada lote se aplica con `MERGE` por `id` (inserta episodios nuevos, actualiza los que ya existen).

In [0]:
def create_bronze_table():
    """Crea bronze vacía con Change Data Feed activado (así los cambios se registran desde la versión 0)."""
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {BRONZE} (
            {ESQUEMA_EPISODIOS},
            _rescued_data STRING, source_file STRING, ingested_at TIMESTAMP
        ) TBLPROPERTIES (delta.enableChangeDataFeed = true)
    """)


def autoloader_read():
    """LECTURA · Auto Loader detecta en RAW los archivos nuevos y los lee con el esquema explícito."""
    return (spark.readStream.format("cloudFiles")
            .option("cloudFiles.format", "json")
            .option("multiLine", "true")                              # cada archivo es un arreglo JSON
            .option("cloudFiles.schemaLocation", SCHEMA_PATH)
            .option("cloudFiles.rescuedDataColumn", "_rescued_data")  # lo que no encaje en el esquema va aquí
            .schema(ESQUEMA_EPISODIOS)
            .load(RAW_PATH)
            .select("*",
                    F.col("_metadata.file_name").alias("source_file"),
                    F.current_timestamp().alias("ingested_at")))


def upsert_to_bronze(lote, lote_id):
    """Aplica un lote del stream en bronze con MERGE: inserta episodios nuevos y actualiza los que ya existen."""
    from delta.tables import DeltaTable
    lote = lote.dropDuplicates(["id"])                                # MERGE exige una sola fila por clave
    (DeltaTable.forName(lote.sparkSession, qualified_name(BRONZE)).alias("t")
        .merge(lote.alias("s"), "t.id = s.id")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())


def autoloader_write(stream_df):
    """ESCRITURA · cada lote del stream entra a bronze con MERGE (upsert); espera a que termine."""
    query = (stream_df.writeStream
             .foreachBatch(upsert_to_bronze)                          # lote a lote: MERGE en bronze
             .option("checkpointLocation", CHECKPOINT_PATH)
             .trigger(availableNow=True)                              # procesa lo pendiente y se detiene
             .start())
    query.awaitTermination()


def run_bronze():
    """RAW → bronze: lectura + escritura (upsert) con Auto Loader."""
    create_bronze_table()
    antes = table_version(BRONZE)
    autoloader_write(autoloader_read())
    merges = (spark.sql(f"DESCRIBE HISTORY {BRONZE}")
              .where(f"version > {antes} AND operation = 'MERGE'")
              .select("operationMetrics").collect())
    if not merges:
        print("bronze: sin archivos nuevos")
        return
    nuevos = sum(int(m[0].get("numTargetRowsInserted", 0)) for m in merges)
    actualizados = sum(int(m[0].get("numTargetRowsUpdated", 0)) for m in merges)
    print(f"bronze: {nuevos} episodios nuevos, {actualizados} actualizados")

## 6. Change Data Feed: leer cambios y recordar hasta dónde se procesó

In [0]:
def read_changes(tabla, desde, hasta):
    """Lee el Change Data Feed de una tabla entre dos versiones (ambas incluidas)."""
    return (spark.read.format("delta")
            .option("readChangeFeed", "true")
            .option("startingVersion", desde)
            .option("endingVersion", hasta)
            .table(tabla))


def last_processed(step):
    """Hasta qué versión del origen ya procesó este paso (-1 si nunca corrió)."""
    spark.sql(f"CREATE TABLE IF NOT EXISTS {STATE} (step STRING, last_version BIGINT, updated_at TIMESTAMP)")
    fila = spark.table(STATE).where(F.col("step") == step).first()
    return fila["last_version"] if fila else -1


def save_processed(step, version):
    """Guarda hasta qué versión procesó este paso."""
    spark.sql(f"""
        MERGE INTO {STATE} t
        USING (SELECT '{step}' AS step, {version} AS last_version, current_timestamp() AS updated_at) s
        ON t.step = s.step
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)

## 7. Silver: bronze → silver con CDF + MERGE

In [0]:
SILVER_COLS = ["episode_id", "episode_name", "season", "episode_number", "airdate", "runtime_min",
               "episode_rating", "show_id", "show_name", "show_language", "network_name", "source_file"]


def create_silver_table():
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {SILVER} (
            episode_id BIGINT, episode_name STRING, season INT, episode_number INT, airdate DATE,
            runtime_min INT, episode_rating DOUBLE, show_id BIGINT, show_name STRING,
            show_language STRING, network_name STRING, source_file STRING
        ) TBLPROPERTIES (delta.enableChangeDataFeed = true)
    """)


def to_silver(df):
    """Aplana el JSON de bronze: una fila por episodio con tipos correctos."""
    extra = [c for c in ("_change_type", "_commit_version") if c in df.columns]
    return df.select(
        F.col("id").alias("episode_id"),
        F.col("name").alias("episode_name"),
        "season",
        F.col("number").alias("episode_number"),
        F.expr("try_cast(airdate AS DATE)").alias("airdate"),
        F.col("runtime").alias("runtime_min"),
        F.col("rating.average").alias("episode_rating"),
        F.col("_embedded.show.id").alias("show_id"),
        F.col("_embedded.show.name").alias("show_name"),
        F.col("_embedded.show.language").alias("show_language"),
        F.coalesce(F.col("_embedded.show.network.name"),
                   F.col("_embedded.show.webChannel.name"),
                   F.lit("(sin red)")).alias("network_name"),
        "source_file",
        *extra)


def build_silver():
    """bronze → silver: aplica solo los cambios de bronze que aún no se procesaron."""
    desde, hasta = last_processed("silver") + 1, table_version(BRONZE)
    if desde > hasta:
        print("silver al día: no hay cambios nuevos en bronze")
        return

    # 1. Cambios de bronze pendientes (la imagen previa de un UPDATE no se aplica)
    cambios = read_changes(BRONZE, desde, hasta).where("_change_type <> 'update_preimage'")

    # 2. Aplanar y quedarse con el último cambio de cada episodio
    ultimo = Window.partitionBy("episode_id").orderBy(F.col("_commit_version").desc())
    filas = (to_silver(cambios).filter("episode_id IS NOT NULL")
             .withColumn("rn", F.row_number().over(ultimo)).where("rn = 1").drop("rn"))

    # 3. MERGE: delete borra, update actualiza, insert inserta
    create_silver_table()
    valores = {c: f"s.{c}" for c in SILVER_COLS}
    (DeltaTable.forName(spark, qualified_name(SILVER)).alias("t")
        .merge(filas.alias("s"), "t.episode_id = s.episode_id")
        .whenMatchedDelete(condition="s._change_type = 'delete'")
        .whenMatchedUpdate(set=valores)
        .whenNotMatchedInsert(condition="s._change_type <> 'delete'", values=valores)
        .execute())

    # 4. Recordar hasta dónde llegamos
    save_processed("silver", hasta)
    print(f"silver: aplicados {filas.count()} cambios de bronze (versiones {desde} a {hasta})")

## 8. Gold: silver → gold con CDF + MERGE

In [0]:
GOLD_KEYS = ["airdate", "network_name"]


def create_gold_table():
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {GOLD} (
            airdate DATE, network_name STRING, episodes BIGINT, shows BIGINT,
            avg_rating DOUBLE, total_minutes BIGINT
        )
    """)


def build_gold():
    """silver → gold: recalcula solo las redes y días que cambiaron en silver."""
    desde, hasta = last_processed("gold") + 1, table_version(SILVER)
    if desde > hasta:
        print("gold al día: no hay cambios nuevos en silver")
        return

    # 1. Claves (red, día) tocadas por los cambios de silver (incluye las imágenes previas)
    afectados = read_changes(SILVER, desde, hasta).select(*GOLD_KEYS).distinct()

    # 2. Recalcular solo esas claves a partir de silver
    recalculo = (spark.table(SILVER).join(afectados, GOLD_KEYS)
                 .groupBy(*GOLD_KEYS)
                 .agg(F.count("*").alias("episodes"),
                      F.countDistinct("show_id").alias("shows"),
                      F.round(F.avg("episode_rating"), 2).alias("avg_rating"),
                      F.sum("runtime_min").alias("total_minutes")))
    fuente = afectados.join(recalculo, GOLD_KEYS, "left").fillna({"episodes": 0})   # sin filas → 0 → se borra

    # 3. MERGE en gold
    create_gold_table()
    (DeltaTable.forName(spark, qualified_name(GOLD)).alias("t")
        .merge(fuente.alias("s"), "t.airdate <=> s.airdate AND t.network_name <=> s.network_name")
        .whenMatchedDelete(condition="s.episodes = 0")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll(condition="s.episodes > 0")
        .execute())

    # 4. Recordar hasta dónde llegamos
    save_processed("gold", hasta)
    print(f"gold: recalculadas {afectados.count()} claves (red, día) por cambios de silver (versiones {desde} a {hasta})")

## 9. Reiniciar el pipeline

Borra las tablas y la memoria de Auto Loader. **RAW no se toca** (los JSON descargados se conservan).

In [0]:
def reset_pipeline():
    """Deja el pipeline como nuevo: tablas y memoria de Auto Loader. RAW se conserva."""
    for t in (BRONZE, SILVER, GOLD, STATE):
        spark.sql(f"DROP TABLE IF EXISTS {t}")
    for p in (SCHEMA_PATH, CHECKPOINT_PATH):
        dbutils.fs.rm(p, recurse=True)
    print("✔ Pipeline reiniciado (RAW intacto)")

In [0]:
print("✔ Setup listo")
print(f"  Esquema : {CATALOG}.{SCHEMA}")
print(f"  Archivos: {BASE}")
print(f"  Tablas  : {BRONZE} → {SILVER} → {GOLD}")